In [ ]:
import smtplib
import requests
from loguru import logger
import pandas as pd
from constants import ADMIN_TOKEN, MAIL_SENDER_ADDRESS, MAIL_PASSWORD, MAIL_SERVER_URL
from dataclasses import dataclass
from typing import Literal
from email.mime.text import MIMEText

In [ ]:
def send_mail(receiver_email, message):
    with smtplib.SMTP(MAIL_SERVER_URL, 587) as server:
        server.starttls()
        server.login(MAIL_SENDER_ADDRESS, MAIL_PASSWORD)
        server.sendmail(MAIL_SENDER_ADDRESS, receiver_email, message)

In [ ]:
# for local use http://127.0.0.1:5000
API_BASE_URL = "https://api.little-mexican-wedding.info"

In [ ]:
health_response = requests.get(f"{API_BASE_URL}/health")

logger.info(health_response.json())
health_response.raise_for_status()

In [ ]:
session = requests.Session()

In [ ]:
login_response = session.post(f"{API_BASE_URL}/auth/login", json ={"token": ADMIN_TOKEN})

logger.info(login_response.json())
login_response.raise_for_status()

In [ ]:
@dataclass
class UserInfo:
    id: int
    first_name: str | None
    last_name: str | None
    mail: str | None
    language: Literal["de", "en", "es"] | None
    attendance: str
    token: str | None = None

In [ ]:
user_list = [
    UserInfo(u["id"],u["first_name"], u["last_name"], u["mail"], u["language"], u["attendance"])
    for u in session.get(f"{API_BASE_URL}/users").json()
]
for user in user_list:
    user.token = session.get(f"{API_BASE_URL}/user/{user.id}/token").json()["token"]

In [ ]:
@dataclass
class MailContent:
    subject: str
    message: str
    
CONTENT = {
    "de": MailContent(
        "Geschenke für unsere Hochzeit",
        """
        <html>
          <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
            <p>Hallo {first_name},</p>
            <p>Nächste Woche Samstag ist es soweit - wir heiraten! Und wir freuen uns enorm darauf, euch alle in Mérida zu sehen ❤️</p>
            <p>Wie ihr euch vorstellen könnt, möchten wir keine materiellen Geschenken, da unser Gepäck sehr voll sein wird. Und anders als früher haben wir schon einen Haushalt mit allem, was wir brauchen.</p>
            <p>Also fühlt euch bitte nicht verpflichtet, uns etwas zu schenken. Wenn ihr das trotzdem möchtet, ist hier ein personalisierter Link zu unserer Webseite, wo ihr aus einer Liste auswählen könnt, wo ihr etwas beitragen möchtet, und wie viel:<br/>
            <a href="https://little-mexican-wedding.info/gifts?token={token}" style="color: #1a0dab;">https://little-mexican-wedding.info/gifts?token={token}</a></p> 
            <p>Dort findet ihr auch unsere Bankdaten. Ob Massage, ein besonderes Essen oder Geld für Möbel - wir wissen alles sehr zu schätzen, aber das größte Geschenk ist, dich auf unserer Hochzeit zu sehen!</p>
            <p>Wenn ihr Fragen habt, lasst es uns wissen:<br>
            mail: <a href="mailto:hello@little-mexican-wedding.info" style="color: #1a0dab;">hello@little-mexican-wedding.info</a><br>
            Regina: +49 1525 606 2564<br>
            Yannic: +49 179 235 5762</p>
            <p>Mit viel Liebe & Vorfreude,<br>Regina & Yannic</p>
          </body>
        </html>
        """
    ),
    "en": MailContent(
        "Gifts for our Wedding",
        """
        <html>
          <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
            <p>Hi {first_name}</p>
            <p>Next week Saturday, it’s happening - we are getting married! We’re very excited to see you all in Mérida ❤️</p>
            <p>As you can imagine, we don’t want any physical gifts, as our bags won’t fit more than what we are traveling with already - and unlike in the old days, we already have a shared household, with all necessities covered.</p>
            <p>Please do not feel obliged to offer a gift. If you want to, here is a personalised link to our website, where you can select from a list what you would like to contribute to, and how much:<br/>
            <a href="https://little-mexican-wedding.info/gifts?token={token}" style="color: #1a0dab;">https://little-mexican-wedding.info/gifts?token={token}</a></p>
            <p>You’ll also find our bank details there. Whether it’s a couple‘s massage, a nice dinner or money for furniture - we appreciate everything, but the greatest gift will be to see you at our wedding!</p>
            <p>If you have any questions, please let us know:<br/>
            mail: <a href="mailto:hello@little-mexican-wedding.info" style="color: #1a0dab;">hello@little-mexican-wedding.info</a><br>
            Regina: +49 1525 606 2564<br>
            Yannic: +49 179 235 5762</p>
            <p>With love & antcipation,<br/>Regina & Yannic</p>
          </body>
        </html>
        """
    ),
    "es" :MailContent(
        "Regalos para nuestra boda",
        """
        <html>
          <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
            <p>Hola {first_name},</p>
            <p>¡El próximo Sábado nos casamos! Estamos muy emocionados de verlos a todos en Mérida ❤️</p>
            <p>Como te imaginarás, no queremos regalos físicos, ya que nuestras maletas no tienen espacio para más cosas de las que ya llevamos —y, a diferencia de antes, ya tenemos un hogar con todo lo necesario.</p>        
            <p>Por favor, no sientas ninguna obligación a regalarnos algo. Si aun así deseas hacerlo, aquí tienes un enlace personal a nuestra página web, donde puedes elegir de una lista a qué te gustaría contribuir y con cuánto:<br/>
            <a href="https://little-mexican-wedding.info/gifts?token={token}" style="color: #1a0dab;">https://little-mexican-wedding.info/gifts?token={token}</a></p>
            <p>Ahí también encontrarás nuestros datos bancarios. Ya sea un masaje en pareja, una cena rica o una aportación para muebles, lo apreciamos muchísimo, pero el mayor regalo será compartir este día contigo en nuestra boda.</p>
            <p>Si tienes alguna pregunta, no dudes en contactarnos:<br>
            mail: <a href="mailto:hello@little-mexican-wedding.info" style="color: #1a0dab;">hello@little-mexican-wedding.info</a><br>
            Regina: +49 1525 606 2564<br>
            Yannic: +49 179 235 5762</p>
            <p>Con mucho cariño y emoción,<br>
            Regina & Yannic</p>
          </body>
        </html>
        """
    )
}

In [ ]:
def send_user_mail(user: UserInfo):    
    if user.attendance == "will_not_join":
        logger.success(f"skipping - user will not attend: {user.first_name} {user.last_name}")
        return
    if user.mail == None:
        logger.warning(f"skipping - no mail set: {user.first_name} {user.last_name}")
        return
        
    language = user.language
    if language is None:
        language = "en"
        logger.warning(f"using default language 'en': {user.first_name} {user.last_name}")

    mail_content = CONTENT[language]
    msg = MIMEText(
        mail_content.message.format(first_name=user.first_name, token=user.token),
        "html", "utf-8"
    )

    msg["Subject"] = mail_content.subject 
    msg['From'] = MAIL_SENDER_ADDRESS
    msg['To'] = user.mail

    try:
        #send_mail(user.mail, msg.as_string())
        logger.success(f"mail sent: {user.first_name} {user.last_name}")
    except Exception as e:
        logger.error(f"mail failed: {user.first_name} {user.last_name}")
        logger.error(e)

for user in user_list:
    send_user_mail(user)